In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
train_gen = ImageDataGenerator(
    rescale=1/255,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.15,
    horizontal_flip=False,
)

In [5]:
val_data = train_gen.flow_from_directory(
    "/content/drive/MyDrive/x ray dataset/xray_dataset_covid19/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    subset="validation"
)

Found 28 images belonging to 2 classes.


In [6]:
train_data = train_gen.flow_from_directory(
    "/content/drive/MyDrive/x ray dataset/xray_dataset_covid19/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    subset="training"
)

Found 120 images belonging to 2 classes.


In [7]:
test_gen = ImageDataGenerator(rescale=1/255)
test_data = test_gen.flow_from_directory(
    "/content/drive/MyDrive/x ray dataset/xray_dataset_covid19/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"
)

Found 40 images belonging to 2 classes.


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers  import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

In [9]:
print(train_data.class_indices)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [10]:
model = Sequential()

In [11]:
model.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, (3,3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(128, (3,3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(1, activation="sigmoid"))

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,631,169 (21.48 MB)

 Trainable params: 5,631,169 (21.48 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor="val_loss", patience=5)

In [14]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [15]:
history = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=[early_stop])

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 49s 12s/step - accuracy: 0.5333 - loss: 1.0974 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 26s 7s/step - accuracy: 0.5500 - loss: 0.6600 - val_accuracy: 0.5000 - val_loss: 0.6532
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 4s/step - accuracy: 0.6833 - loss: 0.6059 - val_accuracy: 0.7500 - val_loss: 0.6138
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 4s/step - accuracy: 0.8000 - loss: 0.5114 - val_accuracy: 0.6071 - val_loss: 0.6370
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 5s/step - accuracy: 0.8083 - loss: 0.4541 - val_accuracy: 0.7143 - val_loss: 0.5171
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 5s/step - accuracy: 0.8833 - loss: 0.3287 - val_accuracy: 0.7857 - val_loss: 0.4344
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 19s 4s/step - accuracy: 0.8667 - loss: 0.3215 - val_accuracy: 0.9286 - val_loss: 0.2954
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 4s/step - accuracy: 0.8583 - loss: 0.2976 - val_accuracy: 0.8214 - val_loss: 0.3190
Epoch 9

In [16]:
model.save("pneumonia_prediction.keras")

In [17]:
from tensorflow.keras.models import load_model
model = load_model("pneumonia_prediction.keras")

In [18]:
import cv2
import numpy as np
image = cv2.imread("/content/drive/MyDrive/x ray dataset/xray_dataset_covid19/single prediction/normal_or_pneumonia1.jpeg")
image = cv2.resize(image, (224,224))
image = image/255.0
image = np.expand_dims(image, axis=0)

In [19]:
prediction = model.predict(image)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
[[0.20468464]]


In [20]:
prob = prediction[0][0]

if prob > 0.5:
    print(f"Prediction: Pneumonia ({prob*100:.2f}% confidence)")
else:
    print(f"Prediction: Normal ({(1-prob)*100:.2f}% confidence)")

Prediction: Normal (79.53% confidence)
